In [1]:
# Install AutoGluon
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking

In [2]:
# Imports

import pandas as pd
import numpy as np
from autogluon.tabular import TabularDataset, TabularPredictor
import time
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("AutoGluon Version:", TabularPredictor.__module__)

AutoGluon Version: autogluon.tabular.predictor.predictor


In [3]:
# Load Data

df = pd.read_csv("athletes_clean.csv")

In [10]:
# Drop Columns

df = df.drop(columns=['Unnamed: 0', 'athlete_id'], axis=1)

In [15]:
# Run AutoML

target = "total lift"
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

predictor = TabularPredictor(label=target, eval_metric='r2').fit(train_data, presets='best_quality',
    excluded_model_types=['CAT', 'NN'],  time_limit=300)

No path specified. Models will be saved in: "AutogluonModels/ag-20251112_234325"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       10.52 GB / 12.67 GB (83.0%)
Disk Space Avail:   62.94 GB / 107.72 GB (58.4%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_l

(_ray_fit pid=6342) [1000]	valid_set's l2: 110.533	valid_set's r2: 0.998581


(_ray_fit pid=6343) 	Ran out of time, early stopping on iteration 1006. Best iteration is:
(_ray_fit pid=6343) 	[1006]	valid_set's l2: 113.809	valid_set's r2: 0.998517
(_ray_fit pid=6475) 	Ran out of time, early stopping on iteration 911. Best iteration is: [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(_ray_fit pid=6475) 	[911]	valid_set's l2: 58.123	valid_set's r2: 0.99925 [repeated 2x across cluster]
(_ray_fit pid=6609) 	Ran out of time, early stopping on iteration 813. Best iteration is: [repeated 2x across cluster]
(_ray_fit pid=6609) 	[809]	valid_set's l2: 56.9266	valid_set's r2: 0.999254 [repeated 2x across cluster]
(_ray_fit pid=6738) 	Ran out of time, early stopping on iteration 657. Best iteration is: [repeated 2x across cluster]
(_ray_fit pid=6738) 	[655]	valid_set's l2: 78.258	vali

In [16]:
leaderboard = predictor.leaderboard(test_data, silent=True)
print(leaderboard)

                 model  score_test  score_val eval_metric  pred_time_test  \
0  WeightedEnsemble_L3    0.999688   0.999175          r2       28.511146   
1    LightGBMXT_BAG_L1    0.999680   0.999102          r2       24.072524   
2  WeightedEnsemble_L2    0.999680   0.999102          r2       24.075264   
3    LightGBMXT_BAG_L2    0.999614   0.998895          r2       28.508725   

   pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  \
0      33.611115  207.133007                 0.002420                0.001400   
1      29.165921  142.011379                24.072524               29.165921   
2      29.168658  142.023975                 0.002739                0.002738   
3      33.609714  207.091244                 4.436201                4.443794   

   fit_time_marginal  stack_level  can_infer  fit_order  
0           0.041763            3       True          4  
1         142.011379            1       True          1  
2           0.012596            2 

In [25]:
best_model_name = predictor.model_best
print("Best model:", best_model_name)

sampled_data = test_data.sample(n=200, random_state=42)
feature_importance = predictor.feature_importance(
    data=sampled_data,
    model=best_model_name
)
print(feature_importance)

Computing feature importance via permutation shuffling for 14 features using 200 rows with 5 shuffle sets...


Best model: WeightedEnsemble_L3


	111.92s	= Expected runtime (22.38s per shuffle set)
	55.92s	= Actual runtime (Completed 5 of 5 shuffle sets)


              importance    stddev   p_value  n  p99_high   p99_low
deadlift    2.429088e-01  0.024895  0.000013  5  0.294167  0.191651
backsq      1.903613e-01  0.015885  0.000006  5  0.223069  0.157654
candj       7.726449e-02  0.006716  0.000007  5  0.091093  0.063436
snatch      5.352589e-02  0.005173  0.000010  5  0.064178  0.042874
gender      1.177306e-03  0.000400  0.001384  5  0.002002  0.000353
weight      1.414896e-04  0.000073  0.006103  5  0.000291 -0.000008
experience  8.613548e-05  0.000176  0.167694  5  0.000449 -0.000276
height      2.479825e-05  0.000009  0.001663  5  0.000043  0.000007
age         2.334637e-05  0.000021  0.031774  5  0.000066 -0.000019
schedule    1.701565e-05  0.000019  0.060035  5  0.000057 -0.000023
eat         3.237239e-06  0.000012  0.291478  5  0.000028 -0.000022
howlong     6.481090e-07  0.000011  0.451231  5  0.000024 -0.000022
background -1.007585e-05  0.000003  0.998761  5 -0.000003 -0.000017
region     -1.740341e-05  0.000017  0.959994  5 

Top 3 Features

In [27]:
top_features = feature_importance.index[:3].tolist()
print("Top 3 features:", top_features)

Top 3 features: ['deadlift', 'backsq', 'candj']


In [29]:
predictor_top = TabularPredictor(label=target, eval_metric='r2').fit(
    train_data[top_features + [target]],
    time_limit=300,
    presets='medium_quality',
    excluded_model_types=['CAT', 'NN'],
)

leaderboard_top = predictor_top.leaderboard(
    test_data[top_features + [target]],
    silent=True
)

print(leaderboard_top)

No path specified. Models will be saved in: "AutogluonModels/ag-20251113_000321"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       10.01 GB / 12.67 GB (79.0%)
Disk Space Avail:   62.83 GB / 107.72 GB (58.3%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/content/AutogluonModels/ag-20251113_000321"
Train Data Rows:    24128
Train Data Columns: 3
Label Column:       total lift
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (2135.0, 4.0, 1016.95031, 277.36815)
	If 'regression' is not the correct prob

[1000]	valid_set's l2: 305.508	valid_set's r2: 0.996149
[2000]	valid_set's l2: 284.844	valid_set's r2: 0.996409
[3000]	valid_set's l2: 276.378	valid_set's r2: 0.996516
[4000]	valid_set's l2: 271.567	valid_set's r2: 0.996577
[5000]	valid_set's l2: 269.25	valid_set's r2: 0.996606
[6000]	valid_set's l2: 267.824	valid_set's r2: 0.996624
[7000]	valid_set's l2: 266.282	valid_set's r2: 0.996643
[8000]	valid_set's l2: 266.282	valid_set's r2: 0.996643
[9000]	valid_set's l2: 265.456	valid_set's r2: 0.996654
[10000]	valid_set's l2: 265.257	valid_set's r2: 0.996656


	0.9967	 = Validation score   (r2)
	12.78s	 = Training   runtime
	1.49s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 285.18s of the 285.18s of remaining time.
	Fitting with cpus=1, gpus=0, mem=0.0/10.1 GB


[1000]	valid_set's l2: 277.687	valid_set's r2: 0.9965
[2000]	valid_set's l2: 270.467	valid_set's r2: 0.996591


	0.9966	 = Validation score   (r2)
	5.36s	 = Training   runtime
	0.42s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 279.22s of the 279.22s of remaining time.
	Fitting with cpus=2, gpus=0, mem=0.0/10.1 GB
	0.9946	 = Validation score   (r2)
	8.86s	 = Training   runtime
	0.26s	 = Validation runtime
Fitting model: ExtraTreesMSE ... Training model for up to 266.74s of the 266.74s of remaining time.
	Fitting with cpus=2, gpus=0, mem=0.0/10.0 GB
	0.9959	 = Validation score   (r2)
	4.19s	 = Training   runtime
	0.23s	 = Validation runtime
Fitting model: NeuralNetFastAI ... Training model for up to 257.82s of the 257.81s of remaining time.
	Fitting with cpus=1, gpus=0, mem=0.0/9.9 GB
	0.9971	 = Validation score   (r2)
	20.63s	 = Training   runtime
	0.04s	 = Validation runtime
Fitting model: XGBoost ... Training model for up to 237.12s of the 237.11s of remaining time.
	Fitting with cpus=1, gpus=0
	0.9957	 = Validation score   (r2)
	1.49s	 = Training   runtime

                 model  score_test  score_val eval_metric  pred_time_test  \
0  WeightedEnsemble_L2    0.997174   0.997140          r2        7.225954   
1      NeuralNetFastAI    0.997125   0.997079          r2        0.109364   
2           LightGBMXT    0.996897   0.996659          r2        3.809871   
3        LightGBMLarge    0.996791   0.995988          r2        0.321152   
4        ExtraTreesMSE    0.996626   0.995888          r2        1.957934   
5             LightGBM    0.996580   0.996596          r2        1.345094   
6       NeuralNetTorch    0.996501   0.996370          r2        0.023185   
7      RandomForestMSE    0.996397   0.994570          r2        1.697127   
8              XGBoost    0.996375   0.995715          r2        0.131558   

   pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  \
0       2.178956  43.034322                 0.003691                0.000641   
1       0.039681  20.631958                 0.109364                0

In [30]:
# Sort leaderboard by 'fit_time' to find top 3 fastest models
top_speed_all = leaderboard.sort_values("fit_time").head(3)
top_speed_top_features = leaderboard_top.sort_values("fit_time").head(3)

print("Top 3 fastest models (all features):\n", top_speed_all)
print("Top 3 fastest models (top features):\n", top_speed_top_features)


Top 3 fastest models (all features):
                  model  score_test  score_val eval_metric  pred_time_test  \
1    LightGBMXT_BAG_L1    0.999680   0.999102          r2       24.072524   
2  WeightedEnsemble_L2    0.999680   0.999102          r2       24.075264   
3    LightGBMXT_BAG_L2    0.999614   0.998895          r2       28.508725   

   pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  \
1      29.165921  142.011379                24.072524               29.165921   
2      29.168658  142.023975                 0.002739                0.002738   
3      33.609714  207.091244                 4.436201                4.443794   

   fit_time_marginal  stack_level  can_infer  fit_order  
1         142.011379            1       True          1  
2           0.012596            2       True          2  
3          65.079865            2       True          3  
Top 3 fastest models (top features):
            model  score_test  score_val eval_metric  pred_